# NB10 — Scaffold Split Implementation and Gate

This notebook creates a **true drug-level scaffold split** using Bemis–Murcko scaffolds,
replacing the pathway-holdout split used in NB00–NB09.

## Why scaffold split?

The earlier `split_ho_pathway` column groups drugs by biological pathway, which leaks
structural information across train/test/OOD. A Bemis–Murcko scaffold split ensures:

1. **No drug appears in both train and test/OOD** (drug-level split)
2. **No shared scaffold family leaks across splits** (structural novelty)
3. The OOD split contains structurally novel scaffold families not seen at all during training

## What gets saved

- `scaffold_split_map.csv` — per-drug scaffold and split assignment
- `obs_with_scaffold_split.csv` — full obs table with new `split_scaffold` column
- `scaffold_split_summary_*.csv` — cell / drug / scaffold counts by split
- `scaffold_split_leakage_check.csv` — explicit drug/scaffold leakage assertions
- `scaffold_baseline_metrics.csv` — simple baseline gate on new split
- `scaffold_split_gate.json` — go/no-go flags for downstream notebooks

## 0) Drive + installs

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!pip -q install rdkit anndata scanpy scikit-learn scipy pandas numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 134.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 151.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.


## 1) Config — all paths and hyperparameters in one place

In [8]:
# ==== exact old-notebook style dataset setup/download block (adapted for NB10) ====
import os
import urllib.request
from pathlib import Path
import random
import numpy as np
import anndata as ad

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Define paths
BASE_DIR = Path("/content/drive/MyDrive/ChemDFM")
DATA_DIR = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results_nb10_scaffold"
ART_DIR = RESULTS_DIR / "artifacts"

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_DIR / "sciplex_complete_middle_subset.h5ad"

# Download if missing
if not DATA_PATH.exists():
    print("Downloading SciPlex dataset (this may take a few minutes)...")
    URL = "https://f003.backblazeb2.com/file/chemCPA-datasets/sciplex_complete_middle_subset.h5ad"
    try:
        urllib.request.urlretrieve(URL, DATA_PATH)
        print(f"Download complete! Saved to: {DATA_PATH}")
    except Exception as e:
        print(f"Download failed: {e}")
        raise
else:
    print(f"Dataset already exists at: {DATA_PATH}")

Download complete! Saved to: /content/drive/MyDrive/ChemDFM/data/sciplex_complete_middle_subset.h5ad


In [9]:
import os
import json
import random
import pickle
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import anndata as ad
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score
from scipy.stats import pearsonr

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


@dataclass
class CFG:
    # ── paths ──────────────────────────────────────────────────────────
    data_path: str = str(DATA_PATH)
    results_dir: str = str(RESULTS_DIR)
    art_dir: str = str(ART_DIR)

    # ── column names (with fallbacks handled in code) ──────────────────
    condition_col: str = "condition"
    cell_col: str = "cell_type"
    dose_col: str = "dose"
    fallback_dose_col: str = "dose_val"
    smiles_candidates: tuple = (
        "canonical_smiles", "smiles", "SMILES",
        "Smiles", "smiles_rdkit", "structure"
    )
    control_label: str = "control"
    new_split_col: str = "split_scaffold"

    # ── split sizes ────────────────────────────────────────────────────
    test_frac: float = 0.15
    ood_frac: float = 0.15

    # ── baseline gate ──────────────────────────────────────────────────
    pca_dim: int = 25


cfg = CFG()

os.makedirs(cfg.results_dir, exist_ok=True)
os.makedirs(cfg.art_dir, exist_ok=True)

print("Config:")
for k, v in asdict(cfg).items():
    print(f"  {k}: {v}")

Config:
  data_path: /content/drive/MyDrive/ChemDFM/data/sciplex_complete_middle_subset.h5ad
  results_dir: /content/drive/MyDrive/ChemDFM/results_nb10_scaffold
  art_dir: /content/drive/MyDrive/ChemDFM/results_nb10_scaffold/artifacts
  condition_col: condition
  cell_col: cell_type
  dose_col: dose
  fallback_dose_col: dose_val
  smiles_candidates: ('canonical_smiles', 'smiles', 'SMILES', 'Smiles', 'smiles_rdkit', 'structure')
  control_label: control
  new_split_col: split_scaffold
  test_frac: 0.15
  ood_frac: 0.15
  pca_dim: 25


## 2) Load data and validate schema

In [10]:
adata = ad.read_h5ad(cfg.data_path)
print(adata)
print("\nobs columns:")
print(list(adata.obs.columns))
print("\nvar columns:")
print(list(adata.var.columns))

AnnData object with n_obs × n_vars = 354640 × 2000
    obs: 'cell_type', 'dose', 'dose_character', 'dose_pattern', 'g1s_score', 'g2m_score', 'pathway', 'pathway_level_1', 'pathway_level_2', 'product_dose', 'product_name', 'proliferation_index', 'replicate', 'size_factor', 'target', 'vehicle', 'batch', 'n_counts', 'dose_val', 'condition', 'drug_dose_name', 'cov_drug_dose_name', 'cov_drug', 'control', 'split_ho_pathway', 'split_tyrosine_ood', 'split_epigenetic_ood', 'split_cellcycle_ood', 'SMILES', 'split_ood_finetuning', 'split_ho_epigenetic', 'split_ho_epigenetic_all', 'split_random'
    var: 'id', 'num_cells_expressed-0-0', 'num_cells_expressed-1-0', 'num_cells_expressed-1', 'gene_id', 'in_lincs', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'all_DEGs', 'hvg', 'lincs_DEGs', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

obs columns:
['cell_type', 'dose', 'dose_character', 'dose_pattern', 'g1s_sc

In [11]:
# ── dose fallback ──────────────────────────────────────────────────────
if cfg.dose_col not in adata.obs.columns:
    if cfg.fallback_dose_col in adata.obs.columns:
        print(f"INFO: dose column '{cfg.dose_col}' not found, using fallback '{cfg.fallback_dose_col}'")
        adata.obs[cfg.dose_col] = adata.obs[cfg.fallback_dose_col]
    else:
        raise ValueError(
            f"FATAL: Neither '{cfg.dose_col}' nor '{cfg.fallback_dose_col}' found in obs. "
            f"Available columns: {list(adata.obs.columns)}"
        )

# ── required columns ───────────────────────────────────────────────────
required_obs = [cfg.condition_col, cfg.cell_col, cfg.dose_col]
missing = [c for c in required_obs if c not in adata.obs.columns]
if missing:
    raise ValueError(f"FATAL: Missing required obs columns: {missing}")
print("Schema check passed. Required obs columns present.")

# ── detect SMILES column ───────────────────────────────────────────────
# SMILES can live in obs (per cell) or in a drug-level metadata table.
# We check obs first, then var, then obs_names-indexed lookups.
smiles_col = None
for cand in cfg.smiles_candidates:
    if cand in adata.obs.columns:
        smiles_col = cand
        print(f"Found SMILES in obs: '{smiles_col}'")
        break

if smiles_col is None:
    # Check uns (common in some h5ad distributions)
    for cand in cfg.smiles_candidates:
        if cand in adata.uns:
            print(f"Found SMILES dict in uns: '{cand}'")
            # Copy to obs so downstream code is uniform
            smiles_col = cand
            adata.obs[smiles_col] = adata.obs[cfg.condition_col].map(
                adata.uns[cand]
            )
            break

if smiles_col is None:
    print(
        "WARNING: No SMILES column found in obs or uns.\n"
        "  Tried columns: " + str(cfg.smiles_candidates) + "\n"
        "  Available obs columns: " + str(list(adata.obs.columns)) + "\n"
        "  Available uns keys: " + str(list(adata.uns.keys())) + "\n"
        "\n"
        "  CONSEQUENCE: Scaffold split will fall back to drug-name hashing.\n"
        "  This is structurally weaker than Bemis-Murcko but still a valid\n"
        "  drug-level split. Set smiles_col=None path is taken below."
    )

print(f"\nsmiles_col = {smiles_col}")

Schema check passed. Required obs columns present.
Found SMILES in obs: 'SMILES'

smiles_col = SMILES


## 3) Build drug-level SMILES table

Every drug should have exactly one canonical SMILES. We derive a per-drug
lookup by taking the first non-null SMILES per drug name.

In [12]:
# Build a tidy drug→smiles table from obs
obs = adata.obs.copy()

# Separate controls from perturbations early
is_control = obs[cfg.condition_col].astype(str) == cfg.control_label
pert_obs = obs[~is_control].copy()
ctrl_obs = obs[is_control].copy()

print(f"Total cells      : {len(obs):,}")
print(f"Perturbed cells  : {len(pert_obs):,}")
print(f"Control cells    : {len(ctrl_obs):,}")
print(f"Unique drugs     : {pert_obs[cfg.condition_col].nunique():,}")
print(f"Unique cell types: {obs[cfg.cell_col].nunique():,}")

all_drugs = sorted(pert_obs[cfg.condition_col].astype(str).unique())
print(f"\nFirst 10 drug names: {all_drugs[:10]}")

if smiles_col is not None:
    drug_smiles = (
        pert_obs[[cfg.condition_col, smiles_col]]
        .rename(columns={cfg.condition_col: "drug", smiles_col: "smiles"})
        .dropna(subset=["smiles"])
        .drop_duplicates(subset=["drug"])
        .set_index("drug")["smiles"]
        .to_dict()
    )
    n_with_smiles = sum(d in drug_smiles for d in all_drugs)
    print(f"\nDrugs with SMILES: {n_with_smiles}/{len(all_drugs)}")
    if n_with_smiles < len(all_drugs) * 0.5:
        print(
            f"WARNING: Fewer than 50% of drugs have SMILES ("
            f"{n_with_smiles}/{len(all_drugs)}). "
            "Will use hash-based fallback for drugs without SMILES."
        )
else:
    drug_smiles = {}
    print("No SMILES column; will use hash-based scaffold assignment for all drugs.")

Total cells      : 354,640
Perturbed cells  : 341,636
Control cells    : 13,004
Unique drugs     : 187
Unique cell types: 3

First 10 drug names: ['2-Methoxyestradiol', 'A-366', 'ABT-737', 'AC480', 'AG-14361', 'AG-490', 'AICAR', 'AMG-900', 'AR-42', 'AT9283']

Drugs with SMILES: 187/187


## 4) Compute Bemis–Murcko scaffolds

For drugs with valid SMILES, we extract the Bemis–Murcko scaffold using RDKit.
For drugs without SMILES (or if RDKit is unavailable), we fall back to
deterministic drug-name hashing, which still gives a valid drug-level split.

In [13]:
# ── Try to import RDKit ────────────────────────────────────────────────
try:
    from rdkit import Chem
    from rdkit.Chem.Scaffolds.MurckoScaffold import MurckoScaffoldSmiles
    RDKIT_AVAILABLE = True
    print("RDKit available — using Bemis-Murcko scaffolds")
except ImportError:
    RDKIT_AVAILABLE = False
    print(
        "WARNING: RDKit not importable. "
        "Run `!pip install rdkit-pypi` in the install cell and restart.\n"
        "Falling back to hash-based scaffold grouping (no structural information)."
    )


def get_scaffold(drug_name: str, smiles: str | None) -> str:
    """Return Bemis-Murcko scaffold SMILES, or a pseudo-scaffold string."""
    if RDKIT_AVAILABLE and smiles is not None:
        try:
            mol = Chem.MolFromSmiles(str(smiles))
            if mol is not None:
                scaf = MurckoScaffoldSmiles(mol=mol)
                # Empty scaffold = no ring system (aliphatic-only drug)
                return scaf if scaf else f"__no_ring__{drug_name}"
        except Exception:
            pass
    # Fallback: hash drug name into one of 20 pseudo-scaffold buckets.
    # This is auditable and deterministic but conveys no chemistry.
    bucket = abs(hash(drug_name)) % 20
    return f"__hash_scaffold_{bucket:02d}"


# Build drug-level scaffold table
drug_scaffold_rows = []
for drug in all_drugs:
    smi = drug_smiles.get(drug, None)
    scaf = get_scaffold(drug, smi)
    drug_scaffold_rows.append({
        "drug": drug,
        "smiles": smi,
        "scaffold": scaf,
        "scaffold_source": "bemis_murcko" if (RDKIT_AVAILABLE and smi is not None) else "hash_fallback",
    })

drug_df = pd.DataFrame(drug_scaffold_rows)
print(f"\nDrug scaffold table shape: {drug_df.shape}")
print(f"Unique scaffolds: {drug_df['scaffold'].nunique()}")
print(f"Scaffold sources: {drug_df['scaffold_source'].value_counts().to_dict()}")
drug_df.head(10)

RDKit available — using Bemis-Murcko scaffolds

Drug scaffold table shape: (187, 4)
Unique scaffolds: 169
Scaffold sources: {'bemis_murcko': 187}


,drug,smiles,scaffold,scaffold_source
0,2-Methoxyestradiol,COc1cc2c(cc1O)CC[C@@H]1[C@@H]2CC[C@]2(C)[C@@H]...,c1ccc2c(c1)CCC1C2CCC2CCCC21,bemis_murcko
1,A-366,COc1cc2c(cc1OCCCN1CCCC1)N=C(N)C21CCC1,C1=Nc2cc(OCCCN3CCCC3)ccc2C12CCC2,bemis_murcko
2,ABT-737,CN(C)CCC(CSc1ccccc1)Nc1ccc(S(=O)(=O)NC(=O)c2cc...,O=C(NS(=O)(=O)c1ccc(NCCSc2ccccc2)cc1)c1ccc(N2C...,bemis_murcko
3,AC480,Cc1c(NC(=O)OCC2COCCN2)cn2ncnc(Nc3ccc4c(cnn4Cc4...,O=C(Nc1cc2c(Nc3ccc4c(cnn4Cc4ccccc4)c3)ncnn2c1)...,bemis_murcko
4,AG-14361,CN(C)Cc1ccc(-c2nc3cccc4c3n2CCNC4=O)cc1,O=C1NCCn2c(-c3ccccc3)nc3cccc1c32,bemis_murcko
5,AG-490,N#C/C(=C\c1ccc(O)c(O)c1)C(=O)NCc1ccccc1,O=C(C=Cc1ccccc1)NCc1ccccc1,bemis_murcko
6,AICAR,NC(=O)c1ncn([C@@H]2O[C@H](CO)[C@@H](O)[C@H]2O)c1N,c1cn(C2CCCO2)cn1,bemis_murcko
7,AMG-900,Cc1csc(-c2nnc(Nc3ccc(Oc4ncccc4-c4ccnc(N)n4)cc3...,c1csc(-c2nnc(Nc3ccc(Oc4ncccc4-c4ccncn4)cc3)c3c...,bemis_murcko
8,AR-42,CC(C)[C@H](C(=O)Nc1ccc(C(=O)NO)cc1)c1ccccc1,O=C(Cc1ccccc1)Nc1ccccc1,bemis_murcko
9,AT9283,O=C(Nc1c[nH]nc1-c1nc2cc(CN3CCOCC3)ccc2[nH]1)NC...,O=C(Nc1c[nH]nc1-c1nc2cc(CN3CCOCC3)ccc2[nH]1)NC...,bemis_murcko


## 5) Assign drug-level scaffold split

### Strategy

We split at the **scaffold family** level, not at the individual drug level.
This is the strongest structural split:

- All drugs sharing a scaffold go to the **same** split (no scaffold leakage)
- OOD gets scaffold families not represented at all in train or test
- Test gets scaffold families not in OOD but absent from train

### Algorithm

1. Group drugs by scaffold → scaffold families
2. Shuffle scaffold families with a fixed seed
3. Assign the last `ood_frac` fraction of scaffold families → OOD
4. Assign the next `test_frac` fraction → test
5. Remaining scaffold families → train

### Controls

Control cells have `condition == 'control'` and are NOT drug-specific.
They are assigned `split_scaffold = 'control'` as a sentinel value.
In downstream notebooks (NB11+), control cells are **used in all splits**
as the reference anchor (matching the convention from NB01–NB09).
This is documented explicitly here to prevent confusion.

In [14]:
rng = np.random.default_rng(SEED)

# Group drugs by scaffold
scaffold_to_drugs = drug_df.groupby("scaffold")["drug"].apply(list).to_dict()
all_scaffolds = sorted(scaffold_to_drugs.keys())
print(f"Total scaffold families: {len(all_scaffolds)}")

# Shuffle scaffold families
all_scaffolds_arr = np.array(all_scaffolds)
perm = rng.permutation(len(all_scaffolds_arr))
shuffled_scaffolds = all_scaffolds_arr[perm]

n_total = len(shuffled_scaffolds)
n_ood = max(1, int(np.ceil(n_total * cfg.ood_frac)))
n_test = max(1, int(np.ceil(n_total * cfg.test_frac)))
n_train = n_total - n_ood - n_test

assert n_train >= 1, "Too few scaffold families for this split fraction."

train_scaffolds = set(shuffled_scaffolds[:n_train])
test_scaffolds = set(shuffled_scaffolds[n_train : n_train + n_test])
ood_scaffolds = set(shuffled_scaffolds[n_train + n_test :])

print(f"\nScaffold family counts:")
print(f"  train : {len(train_scaffolds)}")
print(f"  test  : {len(test_scaffolds)}")
print(f"  ood   : {len(ood_scaffolds)}")

# Assign split to each drug
def scaffold_to_split(scaf: str) -> str:
    if scaf in train_scaffolds:
        return "train"
    if scaf in test_scaffolds:
        return "test"
    if scaf in ood_scaffolds:
        return "ood"
    raise ValueError(f"Scaffold not in any set: {scaf}")

drug_df["split_scaffold"] = drug_df["scaffold"].map(scaffold_to_split)

print("\nDrug counts by split:")
print(drug_df["split_scaffold"].value_counts())

# Sanity: no drug in two splits
assert drug_df.groupby("drug")["split_scaffold"].nunique().max() == 1, \
    "FATAL: A drug appears in more than one split!"
print("\nDrug-level uniqueness check passed.")

Total scaffold families: 169

Scaffold family counts:
  train : 117
  test  : 26
  ood   : 26

Drug counts by split:
split_scaffold
train    132
test      28
ood       27
Name: count, dtype: int64

Drug-level uniqueness check passed.


In [15]:
# Map split back to obs
drug_to_split = drug_df.set_index("drug")["split_scaffold"].to_dict()

def assign_obs_split(condition_val: str) -> str:
    cond = str(condition_val)
    if cond == cfg.control_label:
        return "control"   # sentinel — controls are shared across all splits
    return drug_to_split.get(cond, "drop")

adata.obs[cfg.new_split_col] = adata.obs[cfg.condition_col].astype(str).map(assign_obs_split)

split_counts = adata.obs[cfg.new_split_col].value_counts()
print("Cell counts by split_scaffold:")
print(split_counts)

# Verify no 'drop' rows except empty
n_drop = (adata.obs[cfg.new_split_col] == "drop").sum()
if n_drop > 0:
    print(f"\nWARNING: {n_drop} cells assigned 'drop' (drug not in drug_df). "
          "These will be excluded downstream.")

for req_split in ["train", "test", "ood"]:
    assert req_split in split_counts.index and split_counts[req_split] > 0, \
        f"FATAL: Split '{req_split}' has zero cells!"
print("\nAll three required splits (train/test/ood) are non-empty. ✓")

Cell counts by split_scaffold:
split_scaffold
train      240527
test        51243
ood         49866
control     13004
Name: count, dtype: int64

All three required splits (train/test/ood) are non-empty. ✓


## 6) Leakage checks

We explicitly assert:
1. No drug appears in more than one of {train, test, ood}
2. No scaffold family appears in more than one of {train, test, ood}

In [16]:
# Drug-level leakage check
drug_split_map = (
    adata.obs[adata.obs[cfg.new_split_col] != "control"]
    .groupby(cfg.condition_col)[cfg.new_split_col]
    .unique()
)

drug_in_multiple = drug_split_map[drug_split_map.apply(len) > 1]
if len(drug_in_multiple) > 0:
    raise AssertionError(
        f"FATAL LEAKAGE: {len(drug_in_multiple)} drugs appear in multiple splits:\n"
        f"{drug_in_multiple}"
    )
print("Drug-level leakage check: PASSED — no drug crosses split boundaries. ✓")

# Scaffold-level leakage check
scaf_split = drug_df.groupby("scaffold")["split_scaffold"].unique()
scaf_in_multiple = scaf_split[scaf_split.apply(lambda x: len([v for v in x if v in ["train","test","ood"]]) > 1)]
if len(scaf_in_multiple) > 0:
    raise AssertionError(
        f"FATAL LEAKAGE: {len(scaf_in_multiple)} scaffolds span multiple splits:\n"
        f"{scaf_in_multiple}"
    )
print("Scaffold-level leakage check: PASSED — no scaffold crosses split boundaries. ✓")

# Summarize scaffold/drug overlap
leakage_summary = pd.DataFrame([
    {"check": "drug_in_multiple_splits", "n_violating": len(drug_in_multiple), "passed": len(drug_in_multiple) == 0},
    {"check": "scaffold_in_multiple_splits", "n_violating": len(scaf_in_multiple), "passed": len(scaf_in_multiple) == 0},
])
leakage_summary.to_csv(f"{cfg.results_dir}/scaffold_split_leakage_check.csv", index=False)
print("\nLeakage summary saved.")
print(leakage_summary)

Drug-level leakage check: PASSED — no drug crosses split boundaries. ✓
Scaffold-level leakage check: PASSED — no scaffold crosses split boundaries. ✓

Leakage summary saved.
                         check  n_violating  passed
0      drug_in_multiple_splits            0    True
1  scaffold_in_multiple_splits            0    True


## 7) Summary tables

In [17]:
# Helper: perturbed-only obs (excludes controls)
pert_split_obs = adata.obs[
    adata.obs[cfg.new_split_col].isin(["train", "test", "ood"])
].copy()

# Table 1: cells by split
cell_by_split = adata.obs[cfg.new_split_col].value_counts().rename("n_cells").reset_index()
cell_by_split.columns = ["split", "n_cells"]
print("=== Cells by split ===")
print(cell_by_split)
cell_by_split.to_csv(f"{cfg.results_dir}/summary_cells_by_split.csv", index=False)

# Table 2: unique drugs by split
drug_by_split = (
    pert_split_obs.groupby(cfg.new_split_col)[cfg.condition_col]
    .nunique()
    .rename("n_unique_drugs")
    .reset_index()
)
drug_by_split.columns = ["split", "n_unique_drugs"]
print("\n=== Unique drugs by split ===")
print(drug_by_split)
drug_by_split.to_csv(f"{cfg.results_dir}/summary_drugs_by_split.csv", index=False)

# Table 3: unique scaffolds by split
split_drug_scaffold = drug_df[["drug", "scaffold", "split_scaffold"]].rename(
    columns={"split_scaffold": "split"}
)
scaffold_by_split = (
    split_drug_scaffold[split_drug_scaffold["split"].isin(["train", "test", "ood"])]
    .groupby("split")["scaffold"]
    .nunique()
    .rename("n_unique_scaffolds")
    .reset_index()
)
print("\n=== Unique scaffolds by split ===")
print(scaffold_by_split)
scaffold_by_split.to_csv(f"{cfg.results_dir}/summary_scaffolds_by_split.csv", index=False)

# Table 4: cell types by split
celltype_by_split = (
    pert_split_obs.groupby([cfg.new_split_col, cfg.cell_col])
    .size()
    .rename("n_cells")
    .reset_index()
)
celltype_by_split.columns = ["split", "cell_type", "n_cells"]
print("\n=== Cell types by split (perturbed) ===")
print(celltype_by_split.pivot(index="cell_type", columns="split", values="n_cells").fillna(0).astype(int))
celltype_by_split.to_csv(f"{cfg.results_dir}/summary_celltypes_by_split.csv", index=False)

# Table 5: control cells (these are shared — document exactly)
ctrl_by_celltype = (
    adata.obs[adata.obs[cfg.new_split_col] == "control"]
    .groupby(cfg.cell_col)
    .size()
    .rename("n_control_cells")
    .reset_index()
)
ctrl_by_celltype.columns = ["cell_type", "n_control_cells"]
print("\n=== Control cells by cell type (available for all splits) ===")
print(ctrl_by_celltype)
ctrl_by_celltype.to_csv(f"{cfg.results_dir}/summary_controls_by_celltype.csv", index=False)

=== Cells by split ===
     split  n_cells
0    train   240527
1     test    51243
2      ood    49866
3  control    13004

=== Unique drugs by split ===
   split  n_unique_drugs
0    ood              27
1   test              28
2  train             132

=== Unique scaffolds by split ===
   split  n_unique_scaffolds
0    ood                  26
1   test                  26
2  train                 117

=== Cell types by split (perturbed) ===
split        ood   test   train
cell_type                      
A549       11936  12626   59862
K562       13062  12522   60256
MCF7       24868  26095  120409

=== Control cells by cell type (available for all splits) ===
  cell_type  n_control_cells
0      A549             3287
1      K562             3359
2      MCF7             6358


### Control assignment policy (explicit documentation)

Control cells (`condition == 'control'`) are **not drug-specific** — they are
basal-state cells measured without any perturbation. The scaffold split is
meaningless for them.

**Policy**: Controls receive the sentinel label `split_scaffold = 'control'`.
In downstream notebooks (NB11+), the convention from NB01–NB09 is preserved:
- Per-cell-type control means are computed from **all control cells regardless
  of which perturbed-drug split the cell type appears in**.
- The PCA is fit on **train perturbed + train control** cells only.
- Control cells in test/ood are used only for their mean anchor; they are
  never exposed as "new data" that was hidden.

This matches the convention established in NB00/NB01 and is scientifically
sound because control profiles are not drug-conditional.

## 8) Baseline gate

Run a simple cell-aware mean-delta baseline on the new scaffold splits.
This gate checks whether:
- The scaffold test/OOD splits are feasible (non-degenerate)
- The residual baseline still works on the new split
- R² figures are in a plausible range relative to NB00/NB01 results

This is **not** model training — just the same cell-mean-delta baseline used in NB00.

In [18]:
# Extract expression matrix
X = adata.X
if hasattr(X, "toarray"):
    X = X.toarray()
X = np.asarray(X, dtype=np.float32)

# PCA on train cells only (perturbed + control in train split)
train_pca_mask = adata.obs[cfg.new_split_col].isin(["train", "control"]).values
print(f"PCA fit on {train_pca_mask.sum():,} cells (train perturbed + all controls)")

pca = PCA(n_components=cfg.pca_dim, random_state=SEED)
X_pca = np.zeros((adata.n_obs, cfg.pca_dim), dtype=np.float32)
X_pca[train_pca_mask] = pca.fit_transform(X[train_pca_mask]).astype(np.float32)
X_pca[~train_pca_mask] = pca.transform(X[~train_pca_mask]).astype(np.float32)

print(f"PCA explained variance sum: {pca.explained_variance_ratio_.sum():.4f}")

PCA fit on 253,531 cells (train perturbed + all controls)
PCA explained variance sum: 0.2873


In [19]:
# Per-cell-type control means (from all control cells)
control_mask = (adata.obs[cfg.new_split_col].values == "control")
ctrl_means = {}
for cell in sorted(adata.obs[cfg.cell_col].astype(str).unique()):
    m = control_mask & (adata.obs[cfg.cell_col].astype(str).values == cell)
    if m.sum() == 0:
        raise ValueError(
            f"FATAL: No control cells for cell type '{cell}'. "
            "Cannot compute control anchor."
        )
    ctrl_means[cell] = X_pca[m].mean(axis=0).astype(np.float32)

print("Control cell counts by cell type:")
for cell, m_arr in {
    c: (control_mask & (adata.obs[cfg.cell_col].astype(str).values == c))
    for c in ctrl_means
}.items():
    print(f"  {cell}: {m_arr.sum():,}")

# Build X0 (per-cell-type control mean) and DELTA for all cells
X0_pca = np.stack(
    [ctrl_means[c] for c in adata.obs[cfg.cell_col].astype(str).values]
).astype(np.float32)
DELTA_pca = (X_pca - X0_pca).astype(np.float32)

Control cell counts by cell type:
  A549: 3,287
  K562: 3,359
  MCF7: 6,358


In [20]:
def rowwise_pearson(a: np.ndarray, b: np.ndarray) -> float:
    vals = []
    for i in range(a.shape[0]):
        if np.std(a[i]) < 1e-8 or np.std(b[i]) < 1e-8:
            continue
        vals.append(pearsonr(a[i], b[i])[0])
    return float(np.mean(vals)) if vals else np.nan


def compute_metrics(pred: np.ndarray, true: np.ndarray, x0: np.ndarray) -> dict:
    out = {
        "r2_full": float(r2_score(true.reshape(-1), pred.reshape(-1))),
        "pearson_rowmean": rowwise_pearson(true, pred),
        "mse": float(np.mean((true - pred) ** 2)),
    }
    for k in [20, 50]:
        vals = []
        for i in range(true.shape[0]):
            idx = np.argsort(-np.abs(true[i] - x0[i]))[:k]
            vals.append(r2_score(true[i, idx], pred[i, idx]))
        out[f"r2_top{k}"] = float(np.mean(vals))
    return out


def eval_cell_mean_baseline(split_name: str) -> dict:
    """Cell-aware mean-delta baseline (identical logic to NB00)."""
    split_vals = adata.obs[cfg.new_split_col].values
    condition_vals = adata.obs[cfg.condition_col].astype(str).values
    cell_vals = adata.obs[cfg.cell_col].astype(str).values

    eval_mask = (split_vals == split_name) & (condition_vals != cfg.control_label)
    train_pert_mask = (
        split_vals.isin(["train"]) if hasattr(split_vals, 'isin')
        else np.isin(split_vals, ["train"])
    ) & (condition_vals != cfg.control_label)

    # Per-cell-type mean delta from train perturbed
    cell_mean_delta = {}
    global_mean_delta = DELTA_pca[train_pert_mask].mean(axis=0)
    for cell in ctrl_means:
        m = train_pert_mask & (cell_vals == cell)
        cell_mean_delta[cell] = (
            DELTA_pca[m].mean(axis=0).astype(np.float32)
            if m.sum() > 0
            else global_mean_delta
        )

    eval_idxs = np.where(eval_mask)[0]
    if len(eval_idxs) == 0:
        return {"split": split_name, "n_cells": 0}

    pred = X0_pca[eval_idxs] + np.stack(
        [cell_mean_delta[cell_vals[i]] for i in eval_idxs]
    ).astype(np.float32)
    true = X_pca[eval_idxs]
    x0 = X0_pca[eval_idxs]

    metrics = compute_metrics(pred, true, x0)
    return {"split": split_name, "n_cells": len(eval_idxs), **metrics}


baseline_rows = []
for sp in ["test", "ood"]:
    row = eval_cell_mean_baseline(sp)
    baseline_rows.append(row)
    print(f"\nBaseline [{sp}]: n={row.get('n_cells',0):,} | "
          f"r2_top50={row.get('r2_top50', float('nan')):.4f} | "
          f"r2_top20={row.get('r2_top20', float('nan')):.4f}")

baseline_df = pd.DataFrame(baseline_rows)
baseline_df.to_csv(f"{cfg.results_dir}/scaffold_baseline_metrics.csv", index=False)
print("\nBaseline metrics saved.")


Baseline [test]: n=51,243 | r2_top50=0.5349 | r2_top20=0.4536

Baseline [ood]: n=49,866 | r2_top50=0.5541 | r2_top20=0.4719

Baseline metrics saved.


## 9) Save artifacts

In [21]:
# 1) Save drug scaffold map
drug_df.to_csv(f"{cfg.results_dir}/scaffold_split_map.csv", index=False)
print("Saved: scaffold_split_map.csv")

# 2) Save obs table with new split column (used by NB11+)
obs_out = adata.obs.copy()
# Make sure cell and drug index columns are present for NB11
drug_enc = LabelEncoder()
cell_enc = LabelEncoder()
drug_enc.fit(obs_out[cfg.condition_col].astype(str))
cell_enc.fit(obs_out[cfg.cell_col].astype(str))
obs_out["drug_idx"] = drug_enc.transform(obs_out[cfg.condition_col].astype(str))
obs_out["cell_idx"] = cell_enc.transform(obs_out[cfg.cell_col].astype(str))

obs_out.to_csv(f"{cfg.art_dir}/obs_with_scaffold_split.csv")
print("Saved: obs_with_scaffold_split.csv")

# 3) Save PCA model and X_pca array
np.save(f"{cfg.art_dir}/X_pca_{cfg.pca_dim}d.npy", X_pca)
with open(f"{cfg.art_dir}/pca_model.pkl", "wb") as f:
    pickle.dump(pca, f)
print("Saved: X_pca and pca_model")

# 4) Save control means
with open(f"{cfg.art_dir}/ctrl_means_pca.pkl", "wb") as f:
    pickle.dump(ctrl_means, f)
# Also save gene-space control means for biological validation notebooks
ctrl_means_gene = {}
for cell in sorted(adata.obs[cfg.cell_col].astype(str).unique()):
    m = control_mask & (adata.obs[cfg.cell_col].astype(str).values == cell)
    ctrl_means_gene[cell] = X[m].mean(axis=0).astype(np.float32)
with open(f"{cfg.art_dir}/ctrl_means_gene.pkl", "wb") as f:
    pickle.dump(ctrl_means_gene, f)
print("Saved: ctrl_means_pca + ctrl_means_gene")

# 5) Save encoders
with open(f"{cfg.art_dir}/drug_encoder.pkl", "wb") as f:
    pickle.dump(drug_enc, f)
with open(f"{cfg.art_dir}/cell_encoder.pkl", "wb") as f:
    pickle.dump(cell_enc, f)
print("Saved: drug_encoder + cell_encoder")

# 6) Optionally write back to h5ad (commented by default — uncomment if needed)
# adata.write_h5ad(cfg.data_path.replace('.h5ad', '_scaffold_split.h5ad'))
# print("Saved: updated h5ad with split_scaffold column")

print("\nAll artifacts saved.")

Saved: scaffold_split_map.csv
Saved: obs_with_scaffold_split.csv
Saved: X_pca and pca_model
Saved: ctrl_means_pca + ctrl_means_gene
Saved: drug_encoder + cell_encoder

All artifacts saved.


## 10) Gate decision

Evaluate go/no-go criteria for downstream notebooks:

| Gate | Condition | Required? |
|------|-----------|----------|
| A | Schema valid, controls present for all cell types | Hard |
| B | All three splits non-empty | Hard |
| C | No drug/scaffold leakage | Hard |
| D | Baseline r2_top50 > 0.0 on test/ood | Soft (warns) |
| E | RDKit Bemis–Murcko scaffolds used (not hash fallback) | Soft (warns) |

In [23]:
gate = {}

# small helper so json.dump never crashes on numpy scalar types
def py(x):
    if isinstance(x, (np.bool_,)):
        return bool(x)
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    return x

# Gate A — schema
gate["A_schema"] = py(
    all(c in adata.obs.columns for c in required_obs) and
    all(ctrl_means[c].shape[0] == cfg.pca_dim for c in ctrl_means)
)

# Gate B — splits non-empty
gate["B_splits_nonempty"] = py(
    all(int((adata.obs[cfg.new_split_col] == s).sum()) > 0 for s in ["train", "test", "ood"])
)

# Gate C — no leakage
gate["C_no_leakage"] = py(
    len(drug_in_multiple) == 0 and len(scaf_in_multiple) == 0
)

# Gate D — baseline feasible
test_r2 = baseline_df.loc[baseline_df["split"] == "test", "r2_top50"].values
ood_r2  = baseline_df.loc[baseline_df["split"] == "ood",  "r2_top50"].values
gate["D_baseline_positive"] = py(
    len(test_r2) > 0 and float(test_r2[0]) > 0.0 and
    len(ood_r2)  > 0 and float(ood_r2[0])  > 0.0
)

# Gate E — structural split quality
n_bemis = int((drug_df["scaffold_source"] == "bemis_murcko").sum())
gate["E_bemis_murcko_available"] = py(
    bool(RDKIT_AVAILABLE) and n_bemis >= int(len(all_drugs) * 0.5)
)

print("\n" + "=" * 60)
print("SCAFFOLD SPLIT GATE RESULTS")
print("=" * 60)
for k, v in gate.items():
    status = "PASS ✓" if bool(v) else "FAIL ✗"
    print(f"  {k}: {status}")

hard_gates_ok = bool(gate["A_schema"] and gate["B_splits_nonempty"] and gate["C_no_leakage"])

if not hard_gates_ok:
    raise RuntimeError(
        "One or more HARD gates failed (A/B/C). "
        "Do not proceed to NB11 until these are resolved."
    )

if not bool(gate["D_baseline_positive"]):
    print("\nSOFT WARNING: Baseline r2_top50 <= 0 on some splits. "
          "Check split sizes and control availability.")
if not bool(gate["E_bemis_murcko_available"]):
    print("\nSOFT WARNING: Bemis-Murcko scaffolds not available for majority of drugs. "
          "Hash-based split is structurally weaker. "
          "Install rdkit and ensure SMILES column exists.")

gate["hard_gates_ok"] = hard_gates_ok
gate["n_bemis"] = n_bemis
gate["n_all_drugs"] = int(len(all_drugs))

gate_path = f"{cfg.results_dir}/scaffold_split_gate.json"
with open(gate_path, "w") as f:
    json.dump(gate, f, indent=2)

print(f"\nGate saved to: {gate_path}")


SCAFFOLD SPLIT GATE RESULTS
  A_schema: PASS ✓
  B_splits_nonempty: PASS ✓
  C_no_leakage: PASS ✓
  D_baseline_positive: PASS ✓
  E_bemis_murcko_available: PASS ✓

Gate saved to: /content/drive/MyDrive/ChemDFM/results_nb10_scaffold/scaffold_split_gate.json


## Summary

Run the cell below to print a final human-readable report for this notebook.

In [24]:
print("=" * 60)
print("NB10 — SCAFFOLD SPLIT SUMMARY")
print("=" * 60)
print(f"\nDataset: {cfg.data_path}")
print(f"Total cells: {len(adata):,}")
print(f"SMILES column found: {smiles_col}")
print(f"RDKit available: {RDKIT_AVAILABLE}")
print(f"Scaffold source: {'Bemis-Murcko' if RDKIT_AVAILABLE else 'Hash fallback'}")
print(f"\nScaffold families: {len(all_scaffolds)}")
print(f"  → train: {len(train_scaffolds)} families, {drug_by_split[drug_by_split.split=='train']['n_unique_drugs'].values[0]} drugs")
print(f"  → test : {len(test_scaffolds)} families, {drug_by_split[drug_by_split.split=='test']['n_unique_drugs'].values[0]} drugs")
print(f"  → ood  : {len(ood_scaffolds)} families, {drug_by_split[drug_by_split.split=='ood']['n_unique_drugs'].values[0]} drugs")
print("\nBaseline gate (cell-mean-delta in PCA space):")
for _, row in baseline_df.iterrows():
    print(f"  [{row['split']}] r2_top50={row.get('r2_top50', float('nan')):.4f}  r2_top20={row.get('r2_top20', float('nan')):.4f}  n={int(row['n_cells']):,}")
print(f"\nAll hard gates passed: {hard_gates_ok}")
print(f"Artifacts saved to: {cfg.art_dir}")
print(f"Results saved to  : {cfg.results_dir}")
print("\nNext: NB11_Residual_Training_on_ScaffoldSplit.ipynb")

NB10 — SCAFFOLD SPLIT SUMMARY

Dataset: /content/drive/MyDrive/ChemDFM/data/sciplex_complete_middle_subset.h5ad
Total cells: 354,640
SMILES column found: SMILES
RDKit available: True
Scaffold source: Bemis-Murcko

Scaffold families: 169
  → train: 117 families, 132 drugs
  → test : 26 families, 28 drugs
  → ood  : 26 families, 27 drugs

Baseline gate (cell-mean-delta in PCA space):
  [test] r2_top50=0.5349  r2_top20=0.4536  n=51,243
  [ood] r2_top50=0.5541  r2_top20=0.4719  n=49,866

All hard gates passed: True
Artifacts saved to: /content/drive/MyDrive/ChemDFM/results_nb10_scaffold/artifacts
Results saved to  : /content/drive/MyDrive/ChemDFM/results_nb10_scaffold

Next: NB11_Residual_Training_on_ScaffoldSplit.ipynb
